# Data Inspection — Slay the Spire Run Logs

Goal: The aim of this notebook is to understand the structure of the raw dataset before building any processed tables or models.

Source: There exists a community-shared dataset (77M+ runs), which is posted on Reddit, and shared in Google Drive by the STS internal metrics maintainer. This notebook inspects one file: `2020-09-28-04-45_965.json.gz`,containing 965 individual runs.

## 1. Loading the data

The file is a `.json.gz`, a JSON file compressed with gzip. Python's `gzip` module can decompress and read it in one step, without having to manually unzip it first.

In [1]:
import gzip
import json

with gzip.open("../data/raw/2020-09-28-04-45_965.json.gz", "rt", encoding="utf-8") as f:
    runs = json.load(f)

print(type(runs))
print(len(runs))


<class 'list'>
965


## 2. Structure of a single run

The dataset isn't a flat table, it's nested. Before writing any real code, we check the shape at each level rather than assuming that we know the structure.

`runs` is a list of 965 entries. Each entry turned out to be a dict with a single key, `'event'`, which contains the actual run data (50 fields).

In [2]:
first_run = runs[0]
print(type(first_run), first_run.keys())

event = first_run['event']
print(type(event))
print(len(event.keys()))
print(list(event.keys()))

<class 'dict'> dict_keys(['event'])
<class 'dict'>
50
['gold_per_floor', 'floor_reached', 'playtime', 'items_purged', 'score', 'play_id', 'local_time', 'is_ascension_mode', 'campfire_choices', 'neow_cost', 'seed_source_timestamp', 'circlet_count', 'master_deck', 'special_seed', 'relics', 'potions_floor_usage', 'damage_taken', 'seed_played', 'potions_obtained', 'is_trial', 'path_per_floor', 'character_chosen', 'items_purchased', 'campfire_rested', 'item_purchase_floors', 'current_hp_per_floor', 'gold', 'neow_bonus', 'is_prod', 'is_daily', 'chose_seed', 'campfire_upgraded', 'win_rate', 'timestamp', 'path_taken', 'build_version', 'purchased_purges', 'victory', 'max_hp_per_floor', 'card_choices', 'player_experience', 'relics_obtained', 'event_choices', 'is_beta', 'boss_relics', 'items_purged_floors', 'is_endless', 'potions_floor_spawned', 'killed_by', 'ascension_level']


`event` contains run-level data: game state over time (HP, gold, path taken), outcome info (`victory`, `floor_reached`, `killed_by`), and decision data (`character_chosen`, `card_choices`, `campfire_choices`, `event_choices`). 

The field most relevant to this project is `card_choices`. It records, at each decision point, which cards were offered and which one was picked.

## 3. The `card_choices` field

This is the core field for the project. It records every card selection decision made during a run. Understanding its structure closely is essential before any table can be built in Phase 1.

In [3]:
card_choices = event['card_choices']
print(type(card_choices))
print(len(card_choices))
print(card_choices[0])

<class 'list'>
12
{'not_picked': ['Sunder', 'Stack'], 'picked': 'Hologram', 'floor': 1.0}


Each entry is a dict with three fields:
- `floor`: which floor the decision happened
- `picked`: the card that the player added to his deck
- `not_picked`: the cards that were not chosen by the player

Note that the full offered set of cards is `not_picked + picked`

In [4]:
picks = [c['picked'] for c in card_choices]
print(set(picks))

offer_sizes = [len(c['not_picked']) for c in card_choices]
print(set(offer_sizes))


{'Hologram', 'Redo', 'SKIP', 'Ball Lightning', 'Streamline', 'Static Discharge', 'Sunder', 'Melter', 'Consume+1', 'Electrodynamics'}
{2, 3}


Two things worth noting:
- `picked` can be `'SKIP'`. The player may have decided to not pick any of the offered cards. This needs to be handled as a real category and not an error.
- The number of offered cards vary (3 or 4). `len(c['not_picked'])` is 2 or 3, depending on relics, events etc. This means that the offered set cannot be stored in fixed-width columns and motivates a long-format table (one row per offered card, rather than one row per decision) in Phase 1.

## 4. Checking consistency across runs

Everything above was inspected from a single run (`runs[0]`). Before assuming this structure holds throughout the file, it's worth checking a sample of runs.

In [5]:
sample = runs[:20]

for i, r in enumerate(sample):
    event = r['event']
    print(i, event.get('character_chosen'), event.get('floor_reached'), event.get('victory'), len(event.get('card_choices', [])))

0 DEFECT 24 False 12
1 DEFECT 54 False 28
2 IRONCLAD 5 False 2
3 IRONCLAD 10 False 4
4 THE_SILENT 7 False 2
5 DEFECT 31 False 14
6 IRONCLAD 33 False 16
7 THE_SILENT 23 False 11
8 WATCHER 33 False 13
9 IRONCLAD 24 False 10
10 DEFECT 0 False 0
11 IRONCLAD 25 False 12
12 DEFECT 7 False 3
13 IRONCLAD 1 False 0
14 IRONCLAD 17 False 7
15 IRONCLAD 50 False 23
16 IRONCLAD 19 False 7
17 THE_SILENT 22 False 14
18 IRONCLAD 54 False 28
19 DEFECT 56 True 25


This sample of 20 runs revealed two things Phase 1 needs to account for:

- Multiple characters appear in the file (`IRONCLAD`, `DEFECT`, `THE_SILENT`, `WATCHER`), whcih confirms the decision to scope the first pipeline pass to a single character (Ironclad).
- Some runs have zero card choices (e.g. a run ending at floor 0 or 1). These contribute nothing to the card-selection dataset and should be filtered out in Phase 1.

## 5. Other relevant fields

Not every field needs deep inspection, since some fields are clearly metadata (timestamps, version flags) and don't need exploring further. The fields below are the ones most likely to matter as features or context for the card-selection model.

In [6]:
print(event['floor_reached'])
print(len(event['path_per_floor']), event['path_per_floor'][:5])
print(len(event['current_hp_per_floor']), event['current_hp_per_floor'][:5])
print(event['relics'][:5])
print(event['master_deck'][:5])
print(event['character_chosen'])
print(event['victory'])

56
56 ['M', 'M', '?', 'M', 'M']
55 [55, 55, 55, 54, 42]
['Cracked Core', 'Golden Idol', 'Toy Ornithopter', 'Happy Flower', 'DataDisk']
['AscendersBane', 'Zap', 'Dualcast+1', 'Ball Lightning', 'Ball Lightning']
DEFECT
True


- `floor_reached` — how far the run went before ending (24 in this example).
- `path_per_floor` — one entry per floor, room type taken (`M` = monster, `?` = event, etc.). Not used in the current project scope, but structurally similar to `card_choices` (a sequence of decisions), worth noting as a possible extension.
- `current_hp_per_floor` — HP over time. Note: length is `floor_reached + 1`, likely including starting HP — needs confirming before using as a feature.
- `relics` — relics collected, in acquisition order. This is the final list. Reconstructing "relics held at floor X" will require more work.
- `master_deck` — final deck composition. Same caveat as relics: this is an end-of-run snapshot, not a per-decision state.
- `character_chosen` — the class played. Card pools differ substantially by character, so this project is scoped to a single character (Ironclad) to start.
- `victory` — whether the run was won. Not used as a model input directly (would leak future information into earlier decisions), but a candidate for sample weighting later, e.g. weighting decisions from winning runs more heavily, under the idea that they better reflect successful decision-making. Left as an open question for Phase 3.

## 6. Handoff to Phase 1

The fields inspected above mostly describe the *end state* of a run (final deck, final relics, HP trajectory) rather than the state *at the 
moment of each card decision*. Reconstructing that per-decision state, deck size so far, HP at that floor, relics held at that point, is the core task of Phase 1 (dataset construction).

Concretely, Phase 1 needs to:
- Align `card_choices` (by `floor`) with the per-floor state arrays (`current_hp_per_floor`, `gold_per_floor`, `path_per_floor`).
- Reconstruct running deck composition up to each decision point, since `master_deck` only gives the final deck.
- Reconstruct relics held at each point, since `relics` is also a final snapshot.
- Decide how to encode `SKIP` and variable-length offered sets in the long-format table.
- Apply the scope decision: Ironclad only, runs with at least one card choice.

## Summary

- The dataset file contains 965 runs, each wrapped in `[{'event': {...}}]`.
- `event` holds 50 fields: per-floor state arrays, run outcome, and decision data (`card_choices`, `campfire_choices`, `event_choices`).
- `card_choices` is the core field for this project: it records offered vs. picked cards at each decision point, including `SKIP` and variable-length offer sets (3 or 4 cards).
- Scope for the first pipeline pass: Ironclad only, with the win/loss outcome reserved as a possible sample weight (not a feature) for later.
- Next step (Phase 1): construct a long-format table of per-decision state and choices, aligning the snapshot fields above to each decision point.